# Project 1: Object Detection on Traffic Light Dataset

#### Maximum Points: 100

In this project, the goal is to train an object detection model to detect traffic light color using the dataset provided in this notebook. You can use any framework or model, for example YOLO or RT-DETR and inference techniques like SAHI for better detection of small objects. The focus is on creating an accurate and efficient solution for identifying traffic light color.

**The dataset is provided in YOLO format. However, you are free to convert it to any format of your choice according to the library that you are using**

**The dataset contains a total of 5 classes:**

```python
0: 'Green',
1: 'off',
2: 'red',
3: 'wait_on',
4: 'yellow'  
```

## Your Tasks

* Train an object detection model.
* Perform inference on the given video file.
* Share your inference video in an accessible format, such as uploading it to YouTube or providing a shared drive link.
* Provide a video or writeup link explaining your code and approach.

**Alternatively, you can also upload your writeup document to the online lab.**

**Mark Distribution**:

<div align="center">
    <table border="1" style="border-collapse: collapse;">
        <tr>
            <td><h3>No.</h3></td>
            <td><h3>Criteria</h3></td>
            <td><h3>Marks</h3></td>
        </tr>
        <tr>
            <td><h3>1.</h3></td>
            <td><h3> Dataset Visualization</h3></td>
            <td><h3>15</h3></td>
        </tr>
        <tr>
            <td><h3>2.</h3></td>
            <td><h3>mAP50-95 >= 48%</h3></td>
            <td><h3>70</h3></td>
        </tr>
        <tr>
            <td><h3>3.</h3></td>
            <td><h3>Inference on Test Video</h3></td>
            <td><h3>10</h3></td>
        </tr>
        <tr>
            <td><h3>4.</h3></td>
            <td><h3>Writeup/Video Explanation</h3></td>
            <td><h3>5</h3></td>
        </tr>
    </table>
</div>



---
<h2 style = "color: green;">Dataset Download</h2>

The Traffic Light dataset description is given in the README.dataset.txt file inside the Small Traffic Light dataset folder. You can download the zip file of the dataset from the link provided below:

[Traffic Light Dataset](https://www.dropbox.com/scl/fi/v60994ucoillzfzcwe1kl/Small-Traffic-Light.v1i.yolov11.zip?rlkey=b28kal2b8egup5u73vigh953f&st=kqiijqnv&dl=1)

---

**The notebook is divided into multiple grading sections. You have to write code, as mentioned for each section.  For other helper functions, you can write `.py` files and import them in the notebook. You have to submit the notebook along with `.py` files. Your submitted code must be runnable without any bug.**

<h2 style = "color: green">1. Visualize dataset [Points 15]</h2>


**In this sub-section, you have to plot a few images with their bounding box labels overlayed similar to one shown below.**

<img src = "https://www.dropbox.com/scl/fi/iw5c2fcib8cqufnskpfz3/media_dataset_visualization.png?rlkey=ccsl1ntwwh331j3bixnwilgsy&st=hzmq47bb&dl=1" width = "1000" height = "550">


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import cv2

from config import Config
from env import local_env
from find_off_labels import find_all_off_images

In [ ]:
config = Config(local_env, training=False)

In [ ]:
def plot_images_with_bounding_boxes(
        images,
        pred_bounding_boxes=None,
        pred_bounding_boxes_class_labels=None,
        pred_boxes_confidence=None,
        true_bounding_boxes=None,
        true_bounding_boxes_class_labels=None,
        white_text_background=True,
):
    for i, image in enumerate(images):
        h, w = image.shape[:2]
        fig, ax = plt.subplots(1)
        ax.imshow(image)
        ax.axis('off')

        if true_bounding_boxes is not None:
            for j, (x_center, y_center, bw, bh) in enumerate(true_bounding_boxes[i]):
                x0 = (x_center - bw / 2) * w
                y0 = (y_center - bh / 2) * h
                rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                         linewidth=1, edgecolor='orange', facecolor='none')
                ax.add_patch(rect)
                if true_bounding_boxes_class_labels is not None:
                    label = true_bounding_boxes_class_labels[i][j]
                    if white_text_background:
                        ax.text(x0, y0, label, color='orange', fontsize=8, va='bottom', ha='left',
                                bbox=dict(facecolor='white', edgecolor='none', pad=2))
                    else:
                        ax.text(x0, y0, label, color='orange', fontsize=8, va='bottom', ha='left')

        if pred_bounding_boxes is not None:
            for j, (x_center, y_center, bw, bh) in enumerate(pred_bounding_boxes[i]):
                x0 = (x_center - bw / 2) * w
                y0 = (y_center - bh / 2) * h
                rect = patches.Rectangle((x0, y0), bw * w, bh * h,
                                         linewidth=1, edgecolor='blue', facecolor='none')
                ax.add_patch(rect)
                if pred_boxes_confidence is not None:
                    conf_str = f"{int(pred_boxes_confidence[i][j] * 100)}%"
                    if pred_bounding_boxes_class_labels is not None:
                        label = f"{pred_bounding_boxes_class_labels[i][j]} {conf_str}"
                    else:
                        label = conf_str
                    if white_text_background:
                        ax.text(x0, y0, label, color='blue', fontsize=8,va='bottom', ha='left',
                                bbox=dict(facecolor='white', edgecolor='none', pad=2))
                    else:
                        ax.text(x0, y0, label, color='blue', fontsize=8,va='bottom', ha='left')

        plt.tight_layout()
        plt.show()

In [ ]:
def plot_images_with_ground_truth_bounding_boxes(
    image_paths,
    image_labels,
    white_text_background=True,
):
    images = []
    all_boxes = []
    all_class_labels = []
    for image_path, image_label in zip(image_paths, image_labels):
        image = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
        images.append(image)
        boxes = []
        box_labels = []
        with open(image_label, 'r') as label_file:
            for line in label_file:
                line_segments = line.strip().split()
                box_labels.append(config.class_labels[int(line_segments[0])])
                coords = [float(c) for c in line_segments[-4:]]
                boxes.append(coords)
        all_boxes.append(boxes)
        all_class_labels.append(box_labels)
    plot_images_with_bounding_boxes(images, true_bounding_boxes=all_boxes, true_bounding_boxes_class_labels=all_class_labels, white_text_background=white_text_background)

In [ ]:
def plot_val_images_with_ground_truth_bounding_boxes(
    num_images_to_show=3,
    white_text_background=True,
):
    ids = np.random.choice(config.env.fetch_original_val_ids(), num_images_to_show)
    image_paths = []
    image_labels = []
    for image_id in ids:
        image_paths.append(f'{config.env.fixed_val_images_folder}{image_id}.jpg')
        image_labels.append(f'{config.env.fixed_val_labels_folder}{image_id}.txt')
    plot_images_with_ground_truth_bounding_boxes(image_paths, image_labels, white_text_background=white_text_background)

### Correcting mislabeled data

There are only 4 'off' labeled traffic lights in the dataset, and only one is labeled correctly. So, after posting in the course forums and getting instructor approval, I am removing the 'off' class and only training my model to predict the other 4 classes. I fixed the incorrectly labeled 'off' boxes in the dataset as well. Shown below are the 4 images that originally included a traffic light labeled 'off' - 2 in the train set and 2 in the validation set. As you can see, only the 3rd image actually has a traffic light that is off; the other 3 are mislabeled.

In [ ]:
off_image_paths, off_image_labels = find_all_off_images()
plot_images_with_ground_truth_bounding_boxes(off_image_paths, off_image_labels, white_text_background=False)

<h2 style = "color: green;">2. Train the Model </h2>

<h2 style = "color: green;">3. Validate the Model </h2>

<h2 style = "color: green;">4. Inference on given video [10 Points]</h2>

<h3 style><a href = "https://www.dropbox.com/scl/fi/v3uwaid41m5jypk8mhplr/inference_traffic_light_video.mp4?rlkey=u4l9kxa0av81qxcfod06z7e0g&st=ab83a04b&dl=1">Inference Video</a></h3>


<video
     controls
     src="https://www.dropbox.com/scl/fi/v3uwaid41m5jypk8mhplr/inference_traffic_light_video.mp4?rlkey=u4l9kxa0av81qxcfod06z7e0g&st=kp0dxjz3&dl=1"
     width="640">
 </video>

<h2 style = "color: green;">5. Writeup/Video Explanation [5 Points]</h2>

* Give a detailed written description or a video/screen recording explaining your approach and code.
* Once completed upload the notebook to the online lab and submit.
* Provide the link for the explanation and inferenced video in this section.

In [ ]:
#link:
...